<a href="https://colab.research.google.com/github/convictional/decide/blob/llm_generated_kgs/experiments/knowledge_graphs/LLM_Knowledge_Graph_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unstructured Context to Knowledge Graph via LLMs


The main issue for this work is tracked internally and is not public.

This notebook uses a combination of [`instructor`](https://python.useinstructor.com/examples/knowledge_graph/) and [neo4j](https://neo4j.com/docs/python-manual/current/)

We pull in context from decisions being created in the `Decide` App, as well as `Trusted` Guru docs, and some cherry-picked database documents. Testing is being intentionally kept small for now (~25 total docs being parsed) as sprawl is already real and largely due to needed improvements in dedpulication/merging and eliminating erroneous nodes/edges.

# Init the env

In [ ]:
# Langchain is only there for some context parsing, will remove in future
%pip install -U -qqq anthropic openai langchain instructor neo4j

In [ ]:
import os
import pandas as pd
import yaml
from datetime import datetime
from typing import List, Optional

import neo4j
from pydantic import BaseModel, Field
from graphviz import Digraph

from openai import OpenAI
import instructor
from anthropic import Anthropic

from google.colab import auth, drive, userdata
from google.cloud.bigquery import Client, QueryJobConfig
from langchain.text_splitter import HTMLHeaderTextSplitter, RecursiveCharacterTextSplitter


In [ ]:
# Auth in and set env variables
auth.authenticate_user()
drive.mount('/content/drive')

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
NEO4j_AUTH = userdata.get('NEO4j_AUTH')

# Initialize our bq client to run context queries
project_number = '${GCP_PROJECT}'

BQ_CLIENT = Client(project_number)

INSTRUCTOR_CLIENT = instructor.from_anthropic(Anthropic())


# Prompts

## System Prompts

In [ ]:
# Used to generate raw nodes and in-category relationships
INSTRUCTOR_GENERATE_SYSTEM_PROMPT = """
You are an iterative knowledge graph builder. You are given the current state of
the graph, and you must append the nodes and edges to it. Do not produce any
duplcates and try to reuse nodes as much as possible.

Use your skills as an expert graph scientist, with experience working on
business knowledge graphs, to construct a detailed knowledge graph for a
business working in the technology sector, Convictional.com.

Only extract nodes from your assigned node categories as other categories will
be covered by your teammates. A node should only exist in one category, the most
appropriate category. This means that if there is a better fit for a node you're
considering in a different category, use said category if it is in your
assignments, or leave it for your teammate. Your assigned categories are:
{category}

The other categories that will be extracted by your team members are as follows:
{other_categories}

Node labels should be descriptive and unique.
- Manage node labels meticulously. Verify that each node variable declared at
the beginning of a statement is correctly used throughout the query block. Avoid
creating relationships to undefined or blank nodes by checking the correct
propagation of node labels in all relationship definitions.

When identifying and extracting objectives, consider their influence on one
another and try to infer their nature (fundamental or means) based on the
context provided. This involves a thoughtful analysis of the business strategy
as articulated in the context.

Your responsibility is to populate the graph with as many relevant and
accurately detailed nodes and relationships as possible within your designated
categories:
{category}

Never fabricate data.

The current date and time is, {current_datetime}
"""


In [ ]:
PRUNE_NODE_SYSTEM_PROMPT = """
# Instruction
Adopt the persona of a world-class graph scientist with specialized experience in helping
businesses build knowledge graphs for digital twinning. Your expertise in Neo4j and Cypher
query language enables you to effortlessly identify and resolve issues in the code.

Your task is to meticulously review the Cypher code provided by the User. This code often
contains duplications, syntax errors, and disorganization that could hinder its effectiveness.
Your role involves:
- Identifying and merging duplicate nodes or relationships that have identical identifiers or properties.
- Correcting any syntax errors to ensure the code adheres strictly to Cypher language standards.
- Removing any non-Cypher code elements, such as irrelevant code blocks or comments, to clean the script.

Node variables should be descriptive and unique. You may modify node variable
names if required to resolve conflicts or ensure continuity in order to build a
graph structured as intended.
- Manage node variables meticulously. Verify that each node variable declared at
the beginning of a statement is correctly used throughout the query block. Avoid
creating relationships to undefined or blank nodes by checking the correct
propagation of node variables in all relationship definitions.
- Ensure continuity between separate statements. Cypher queries often use
semi-colons to end a command; however, it is crucial to maintain node and
relationship continuity across statements. If a node variable is introduced,
it must be carried forward appropriately to maintain its connection in
subsequent queries unless explicitly terminated.

The goal is to refine the code without altering its underlying content or intent. Ensure that all
changes preserve the original data relationships and node fields as intended by the User.

Please return only the corrected Cypher code, formatted clearly and concisely, so it can be
directly utilized in downstream tasks without further modification.

NOTE: Do not add any comments or extraneous code blocks to your output. The focus is on delivering
clean, valid, and functional Cypher code ready for immediate integration into business applications.
"""


# Context

The context that we'll iteratively build our graph from

## Guru

Using Fivetran, we have Guru syncing into our warehouse and can use the below query to get all not deleted, not archived cards. We can then chunk and embed for retrieval.

In [ ]:
GURU_CARD_QUERY = """
SELECT
  preferred_phrase AS card_title,
  last_verified_by_user AS card_last_verified_by_user,
  cast(date_created AS date) AS card_created_date,
  last_modified AS card_last_modified,
  content AS card_content,
  share_status AS card_share_status,
  verification_interval AS card_verification_interval,
  verification_state AS card_verification_state
FROM
  guru.card
WHERE
  NOT _fivetran_deleted
  AND (
    NOT archived
    OR archived IS NULL
  )
  AND verification_state = 'TRUSTED'
"""

job = BQ_CLIENT.query(GURU_CARD_QUERY)

raw_guru_df = job.to_dataframe()

In [ ]:
# Use langchain (for now) to split up our content on headers
headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
]

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# Define the character splitter for long chunks
chunk_size = 5000  # Example chunk size
chunk_overlap = 300  # Example overlap

character_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)


In [ ]:
# split up card content to be more targeted
# Create an empty DataFrame to hold the processed rows
processed_rows = []

for index, row in raw_guru_df.iterrows():
    # Initial HTML header splitting
    initial_splits = html_splitter.split_text(row['card_content'])

    for document in initial_splits:
        # Further chunk long segments
        chunks = character_splitter.split_text(document.page_content)

        for chunk in chunks:
            # Construct a new row with existing data and the chunked content
            new_row = row.copy()
            new_row['card_content'] = chunk  # Update 'card_content' with chunked text
            new_row['card_header'] = ', '.join([f"{k}: {v}" for k, v in document.metadata.items()])  # Combine metadata into a string

            processed_rows.append(new_row)


# Convert the list of dictionaries to a DataFrame
guru_df = pd.DataFrame(processed_rows).reset_index()

guru_df = guru_df.drop(['index'], axis=1)


## DB Context

In [ ]:
# Specify the path to your .yml file
yml_file_path = '/content/drive/Shareddrives/Customer/Data/Model Assets/Project Obama/context/db_context.yml'

# Load the contents of the .yml file
with open(yml_file_path, 'r') as file:
    yml_data = yaml.safe_load(file)


In [ ]:
# Specify the number of entries you want to access
N = 40

# Access the first dataset
first_dataset = yml_data['datasets'][0]['models']

# Access the first N items from the first dataset
example_db_context = first_dataset[:N]

db_context = pd.DataFrame(example_db_context)

db_context.head()

## Decisions

In [ ]:
DECISIONS_QUERY = """
SELECT
  *
FROM mongo_decide_prod.decisions
where sharing = 'public'
and not _fivetran_deleted
"""

job = BQ_CLIENT.query(DECISIONS_QUERY)

decisions_df = job.to_dataframe()

In [ ]:
decisions_df.head()

# Node Extraction from Context

The LLM doesn't like to be verbose if it doesn't have to and can be conservative in creating nodes across many categories. I think the best approach would be to chain 7 LLMs together and ask each only to extract nodes from their respective category.

Taking this to the extreme, and using the bitter lesson - we could then string 21 LLMs together to evaluate the relationships between each of the 21 pairwise categories.

I need to figure out how to scan all context efficiently

In [ ]:
CLAUDE_HAIKU = "claude-3-haiku-20240307"
CLAUDE_SONNET = "claude-3-sonnet-20240229"
CLAUDE_OPUS = "claude-3-opus-20240229"

OPENAI_GPT4 = "gpt-4-turbo-2024-04-09"

In [ ]:
categories = [
"1. Business Assets - `BusinessAssets`: Similar to assets that one might find accounted for on a balance sheet.",
"2. Business Processes - `BusinessProcesses: These are documented or inferred processes that the business follows in order to operate.",
"3. Customer Related - `CustomerRelated`: These are the individual customers along with their relevant metadata",
"4. Business Team - `BusinessTeam`: These are the departments, teams, working groups, committees and individual team members of the business.",
"5. Business Context - `BusinessContext`: This is the 'ethereal' context of the business and may be broad",
"6. Business Data - `BusinessData`: These are the actual databases, documents or other structured or unstructured data sources",
"7. Business Tools - `BusinessTools`: These are the tools the business uses to operate whether it be software, SaaS, cloud infrastructure, or hardware such as tools, servers, vehicles etc",
"8. Decisions - `Decision`: These are the decisions being considered by the business, along with their relevant context.",
"9. Business Objectives - `BusinessObjective`: These are the strategic goals or business objectives, inspired by decision science principles. Each objective should be classified as either a fundamental objective or a means objective. Fundamental objectives are ends in themselves, while means objectives are steps towards achieving fundamental objectives. You should also brainstorm and list potential measures for each objective, indicating how success can be assessed.",
]

In [ ]:
neo4j_node_categories = [
    "BusinessAssets",
    "BusinessProcesses",
    "CustomerRelated",
    "BusinessTeam",
    "BusinessContext",
    "BusinessData",
    "BusinessTools",
    "Decisions",
    "BusinessObjectives",
]

# Use Neo4j + Instructor

In [ ]:
import json
# Adds response_model to ChatCompletion
# Allows the return of Pydantic model rather than raw JSON

class Node(BaseModel):
    name: str = Field(...,  description="Name of the node. It should be descriptive and unique. Use CamelCase.")
    category: str = Field(..., description=f"One of the following categories: {str(neo4j_node_categories)}")
    description: str = Field(..., description="A short description of the node that is more verbose than the name.")
    source_doc: str = Field(..., description="The name of the source document this node was inferred from. Typically the document or header.")
    source_doc_chunk: str = Field(..., description="The relevant chunk of text from the source document from which this node was inferred.")
    source_doc_location: str = Field(..., description="The link, file path or instruction on how to reach the source document that this node was inferred from.")
    created_at: str = Field(..., description="The datetime the node was created in 'YYYY-MM-DD HH:MM:SS'")
    updated_at: str = Field(..., description="The datetime of when the node was last updated.")
    color: str = Field(..., description="The node category's color. If no nodes in this category have been created yet, choose an unused color; otherwise re-use the current node category color.")

    __hash__ = object.__hash__


class Edge(BaseModel):
    source: str = Field(..., description="Source name of the node that the edge begins from.")
    target: str = Field(..., description="Destination name of the node that the edge terminates in.")
    name: str = Field(..., description="The edge's name representing its meaning")
    source_doc: str = Field(..., description="The name of the source document this edge was inferred from. Typically the document or header.")
    source_doc_chunk: str = Field(..., description="The relevant chunk of text from the source document from which this edge was inferred.")
    source_doc_location: str = Field(..., description="The link, file path or instruction on how to reach the source document that this relationship was inferred from.")
    color: str = Field(default = 'black', description="The edge type's color. If no edges of this type have been created yet, choose an unused color; otherwise re-use the current edge type color.")

    __hash__ = object.__hash__

class KnowledgeGraph(BaseModel):
    nodes: Optional[List[Node]] = Field(..., default_factory=list)
    edges: Optional[List[Edge]] = Field(..., default_factory=list)


    def upsert_node(self, node: Node) -> str:
        props = node.dict()
        props_text = ', '.join(f'n.{k} = ${k}' for k in props if k != 'name')
        query = f"""
        MERGE (n:{node.category.replace(' ', '')} {{name: '{node.name}'}})
        ON CREATE SET {props_text}
        ON MATCH SET {props_text}
        """
        return query, props

    def create_relationship(self, edge: Edge) -> str:
        props = edge.dict()
        props_text = ', '.join(f'r.{k} = ${k}' for k in props if k not in ['source', 'target'])
        query = f"""
        MATCH (a {{name: $source}}), (b {{name: $target}})
        MERGE (a)-[r:{edge.name.replace(' ','_')}]->(b)
        ON CREATE SET {props_text}
        ON MATCH SET {props_text}
        """
        return query, {'source': edge.source, 'target': edge.target, **{k: v for k, v in props.items() if k not in ['source', 'target']}}


    @staticmethod
    def update_graph(graph: 'KnowledgeGraph', updates: 'KnowledgeGraph', session: neo4j.Session):
        with session.begin_transaction() as tx:
            for node in updates.nodes:
                query, params = graph.upsert_node(node)
                tx.run(query, params)
            for edge in updates.edges:
                query, params = graph.create_relationship(edge)
                tx.run(query, params)

    def draw(self, output_path='/content/graph_visualization'):
        dot = Digraph(comment="Knowledge Graph")

        # Add nodes with properties as name for visualization
        for node in self.nodes:
            dot.node(str(node.id), label=node.name, color=node.color)

        # Add edges
        for edge in self.edges:
            dot.edge(str(edge.source), str(edge.target), label=edge.name, color=edge.color)

        # Render graph to a file (PNG, PDF, SVG, etc.)
        dot.render(output_path, format='png', view=True)

    def save_cypher(self, file_path='/content/example_kg.cypher'):
        with open(file_path, 'w') as f:
            for node in self.nodes:
                query, params = self.upsert_node(node)
                f.write(query + ';\n')
            for edge in self.edges:
                query, params = self.create_relationship(edge)
                f.write(query + ';\n')

    def trimmed_graph_json(self):
        """Generates a trimmed version of the graph's JSON representation for prompts."""
        trimmed_nodes = [{'name': node.name, 'category': node.category} for node in self.nodes]
        trimmed_edges = [{'source': edge.source, 'target': edge.target, 'name': edge.name} for edge in self.edges]
        return json.dumps({'nodes': trimmed_nodes, 'edges': trimmed_edges}, indent=2)


In [ ]:
import math

# Methods for generating a knowledge graph given context
def clear_existing_graph(session: neo4j.Session):
    query = "MATCH (n) DETACH DELETE n"
    with session.begin_transaction() as tx:
        tx.run(query)
        print("Cleared existing graph.")

def generate_graph(
    input: List[str],
    session: neo4j.Session,
    clear_graph: bool = False,
    num_parallel_cats: int = 3
    ) -> KnowledgeGraph:

    current_dt = str(datetime.now())
    cur_state = KnowledgeGraph()
    num_iterations = len(input)

    if clear_graph:
        clear_existing_graph(session)

    num_cat_groups = math.ceil(len(categories)/num_parallel_cats)
    for n in range(0, len(categories), num_parallel_cats):
        node_category = categories[n:n + num_parallel_cats]
        other_node_categories = categories[:n] + categories[n + num_parallel_cats:]

        node_category = '\n'.join(node_category)
        other_node_categories = '\n'.join(other_node_categories)

        for i, inp in enumerate(input):
            print(f"""Working on row {i+1} of {num_iterations} rows for node group {n+1} of {num_cat_groups}...""")

            try:
                new_updates = INSTRUCTOR_CLIENT.chat.completions.create(
                    model=CLAUDE_HAIKU,
                    temperature=0.1,
                    max_tokens=4096,
                    messages=[
                        {
                            "role": "system",
                            "content": INSTRUCTOR_GENERATE_SYSTEM_PROMPT.format(
                                category=node_category,
                                other_categories=other_node_categories,
                                current_datetime=current_dt,
                                ),
                        },
                        {
                            "role": "user",
                            "content": f"""
                            ### Current Graph Status
                            Here is the current state of the graph, be sure to
                            add relevant edges or inferred relationships across
                            all categories using the context you'll receive
                            below:
                            {cur_state.trimmed_graph_json()}
                            ### Instruction
                            Use the following piece of context from Convictional
                            and your knowledge to infer nodes and edges from
                            your assigned node categories:
                            {node_category}
                            Be creative!
                            # Part {i}/{num_iterations} of the input:

                            ### Context
                            Incorporate the following context into the knowledge
                            graph:
                            {inp}""",
                        },
                        # {
                        #     "role": "user",
                        #     "content": f"""Here is the current state of the graph, be sure to add relevant edges or inferred relationships across all categories:
                        #     {cur_state.trimmed_graph_json()}""",
                        # },
                    ],
                    response_model=KnowledgeGraph,
                )  # type: ignore

                KnowledgeGraph.update_graph(cur_state, new_updates, session)
                cur_state.draw(output_path=f"/content/drive/Shareddrives/Customer/Data/Model Assets/Project Obama/graphs/iteration_{i}")
            except Exception as ex:
                print(f"Skipped row {i+1} due to an error: {ex}")

    return cur_state


In [ ]:
# Put our context into a list of strings
decisions = [str(row.to_dict()) for _, row in decisions_df.iterrows()]
guru = [str(row.to_dict()) for _, row in guru_df.iterrows()]
database = [str(row.to_dict()) for _, row in db_context.iterrows()][:]

text_chunks = guru + database + decisions

## Generate the graph

In [ ]:
# Init our neo4j connection
URI = "neo4j+s://760faa22.databases.neo4j.io"
AUTH = ("neo4j", NEO4j_AUTH)

with neo4j.GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()

db_connection: Optional[neo4j.Session] = neo4j.GraphDatabase.driver(URI, auth=AUTH).session()

In [ ]:
# Generate a graph given a list of text inputs
graph: KnowledgeGraph = generate_graph(text_chunks, db_connection, clear_graph=True)

# Draw the graph and save to local file
graph.draw(output_path=f"/content/drive/Shareddrives/Customer/Data/Model Assets/Project Obama/graphs/final")

# Save Cypher commands to a file
graph.save_cypher(file_path=f"/content/drive/Shareddrives/Customer/Data/Model Assets/Project Obama/graphs/final_kg.cypher")